# 06 · Unified pipeline — defense in depth

**RQ4.** What happens when the network and host detectors work together?

`src/evaluation/unified_pipeline.py` simulates a monitored environment of **2000 host sessions**. Each session is a
chronological stream of network windows (NIDS graphs) and executed files (PE graphs) drawn from the *test* pools:

| Scenario | Share | Content |
|---|---|---|
| `benign` | 70% | benign traffic + benign software |
| `network_attack` | 10% | malicious traffic only |
| `malware_drop` | 10% | malicious executable only |
| `multi_stage` | 10% | malicious traffic followed by a malicious executable |

Pipeline per session:
1. each event is scored by its MFGraph detector; scores are **calibrated** on held-out data so that the chosen threshold maps to risk 0.5;
2. per-domain max risks are fused into a **unified score** (Noisy-OR boost when both agree, strongest signal otherwise);
3. an alert fires if the unified score ≥ 0.5, with severity NONE / LOW / MEDIUM / HIGH / CRITICAL.

**Impact Mitigation (IM)** — a modelling assumption, not a measured quantity: a threat stopped at the
**network layer counts 1.0** (it never reached the host), at the **host layer 0.5** (it arrived but was blocked before
execution), missed = 0. *Net IM* also subtracts 0.1 per false alert on a benign session to account for analyst cost.
These weights live in `UNIFIED_DEFAULTS["mitigation_weights"]` and should be stated as assumptions in the thesis.

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # torch.jit / pandas deprecations
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm without ipywidgets
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.config import CONFIG

EXP = project_root / CONFIG["experiments_path"]
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})
GNN_NAME = "GNN (MFGraph)"


def latest_run(base: Path, filename: str):
    """Most recent timestamped run folder under `base` that contains `filename`."""
    runs = sorted(p.parent for p in base.glob(f"*/{filename}")) if base.exists() else []
    return runs[-1] if runs else None


def missing(path, task):
    print(f"[Missing] {path}\nRun first:  python orchestrator.py -t {task}")

In [ ]:
run = latest_run(EXP / "unified_pipeline", "summary.json")
if run is None:
    missing(EXP / "unified_pipeline/<run>/summary.json", "unified_pipeline")
else:
    print(f"Using run {run.name}")
    summary = json.loads((run / "summary.json").read_text())
    events = pd.read_csv(run / "events.csv")
    sessions = pd.read_csv(run / "sessions.csv")
    print(f"PE model used: {summary['config']['pe_model']} | thresholds: {summary['thresholds']}")

## 1. Event-level vs. session-level performance

In [ ]:
if run is not None:
    keep = ["auc", "f1", "precision", "recall", "fpr"]
    table = pd.DataFrame({
        "Network windows (NIDS)": summary["event_metrics"]["nids"],
        "Software (PE)": summary["event_metrics"]["pe"],
        "Unified session alert": summary["session_metrics"],
    }).T[keep].astype(float)
    display(table.style.format("{:.3f}"))
    print(f"Impact Mitigation: gross {summary['impact_mitigation']:.1%} | "
          f"net {summary['impact_mitigation_net']:.1%} | false alarm rate {summary['false_alarm_rate']:.1%}")

The question is whether the **session row beats both event rows**. If it does, combining layers adds value; if the
session FPR is higher than either detector alone, the fusion is too permissive (an OR of two detectors adds their false positives).

## 2. Detection rate per scenario

In [ ]:
if run is not None:
    rate = sessions.groupby("scenario")["alert"].mean().mul(100)
    order = ["benign", "network_attack", "malware_drop", "multi_stage"]
    rate = rate.reindex([s for s in order if s in rate.index])
    colors = ["tab:red" if s == "benign" else "tab:blue" for s in rate.index]
    ax = rate.plot(kind="bar", color=colors, edgecolor="black", figsize=(8, 4.5), rot=0)
    ax.set_ylabel("% of sessions alerted"); ax.set_ylim(0, 110)
    ax.set_title("Alert rate per scenario (red = false positive rate on benign sessions)")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontweight="bold")
    plt.tight_layout(); plt.show()

`multi_stage` sessions give the attacker two chances to be caught, so their detection rate should be the highest;
if it is not, the two detectors are missing the *same* sessions (correlated errors).

## 3. Where threats are stopped

In [ ]:
if run is not None:
    threats = sessions[sessions.true_threat == 1]
    stops = pd.crosstab(threats.scenario, threats.stopped_at, normalize="index").mul(100)
    stops = stops.reindex(columns=[c for c in ["network", "host", "missed"] if c in stops.columns])
    ax = stops.plot(kind="barh", stacked=True, figsize=(10, 3.8),
                    color={"network": "#2ca02c", "host": "#ff7f0e", "missed": "#d62728"})
    ax.set_xlabel("% of threat sessions"); ax.set_xlim(0, 100)
    ax.set_title("First layer that stopped each threat")
    ax.legend(["network (IM = 1.0)", "host (IM = 0.5)", "missed (IM = 0)"], bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.show()
    display(pd.crosstab(threats.scenario, threats.stopped_at, margins=True))

For `malware_drop` only the host layer can stop the threat, and for `network_attack` only the network layer. The
interesting row is `multi_stage`: the share stopped at the host layer is the threat that **would have been missed without
defense in depth**.

## 4. Unified score and alert levels

In [ ]:
if run is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
    for lbl, c, txt in [(0, "steelblue", "Benign sessions"), (1, "tomato", "Threat sessions")]:
        axes[0].hist(sessions.loc[sessions.true_threat == lbl, "unified_score"], bins=40, alpha=0.6, color=c, label=txt, density=True)
    axes[0].axvline(summary["config"]["alert_threshold"], ls="--", color="black", label="alert threshold")
    axes[0].set_title("Unified score distribution"); axes[0].set_xlabel("Unified score"); axes[0].legend()

    levels = ["NONE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]
    ct = pd.crosstab(sessions.scenario, sessions.alert_level).reindex(columns=[l for l in levels if l in sessions.alert_level.unique()])
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=axes[1]); axes[1].set_title("Alert level per scenario")
    plt.tight_layout(); plt.show()

**Takeaways to write in the thesis**
* Report gross *and* net IM, and say explicitly that the 1.0 / 0.5 / 0 weights and the 70/10/10/10 scenario mix are design choices; a short sensitivity
  analysis (e.g. host = 0.3 or 0.7, `pe_model = "gan"`) makes the conclusion stronger.
* Benign sessions with HIGH/CRITICAL alerts are the most expensive false positives for an analyst; they are worth discussing separately.
* Sessions reuse graphs from finite test pools, so the simulation measures how detectors **combine**, not new generalisation.